# Wikipedia Redirect Index Demo

这个 notebook 用来加载、检查和测试 `WikipediaRedirectIndex`。

In [ ]:
from pathlib import Path

from wikipedia_redirects import (
    WikipediaRedirectIndex,
    get_bucket_key,
    normalize_wikipedia_title,
)

INDEX_DIR = Path("data/wikipedia_redirects/redirect_index")
INDEX_DIR

In [ ]:
if not INDEX_DIR.exists():
    raise FileNotFoundError(f"Index directory not found: {INDEX_DIR}")

with WikipediaRedirectIndex(INDEX_DIR) as index:
    stats = index.stats()
    metadata = {
        "wiki": index.get_metadata("wiki"),
        "dump_tag": index.get_metadata("dump_tag"),
        "page_dump": index.get_metadata("page_dump"),
        "redirect_dump": index.get_metadata("redirect_dump"),
    }

stats, metadata

## 1. 单个标题查询

In [ ]:
title = "USA"

normalized = normalize_wikipedia_title(title)
bucket_key = get_bucket_key(normalized)

with WikipediaRedirectIndex(INDEX_DIR) as index:
    canonical = index.resolve_redirect(title)
    synonyms = index.get_synonyms(title)

{
    "title": title,
    "normalized": normalized,
    "bucket_key": bucket_key,
    "canonical": canonical,
    "synonym_count": len(synonyms),
    "synonyms": synonyms[:20],
}

## 2. 批量测试多个查询词

In [ ]:
queries = [
    "USA",
    "U.S.A.",
    "US",
    "United States",
    "NYC",
    "New York City",
    "UK",
    "United Kingdom",
]

rows = []
with WikipediaRedirectIndex(INDEX_DIR) as index:
    for query in queries:
        rows.append({
            "query": query,
            "normalized": normalize_wikipedia_title(query),
            "bucket": get_bucket_key(normalize_wikipedia_title(query)),
            "canonical": index.resolve_redirect(query),
            "synonym_count": len(index.get_synonyms(query)),
        })

rows

## 3. 查看某个 canonical 的同义词

In [ ]:
canonical_title = "United States"

with WikipediaRedirectIndex(INDEX_DIR) as index:
    synonyms = index.get_synonyms(canonical_title)

print(f"Canonical: {canonical_title}")
print(f"Total synonyms: {len(synonyms)}")
synonyms[:100]

## 4. 直接查看 bucket 文件内容

In [ ]:
import gzip
import pickle

bucket_to_inspect = "us"
redirect_bucket_path = INDEX_DIR / "redirect_buckets" / f"{bucket_to_inspect}.pkl.gz"
canonical_bucket_path = INDEX_DIR / "canonical_buckets" / f"{bucket_to_inspect}.pkl.gz"

def load_pickle_gz(path: Path):
    with gzip.open(path, "rb") as f:
        return pickle.load(f)

redirect_bucket = load_pickle_gz(redirect_bucket_path) if redirect_bucket_path.exists() else {}
canonical_bucket = load_pickle_gz(canonical_bucket_path) if canonical_bucket_path.exists() else {}

print("redirect bucket size:", len(redirect_bucket))
print("canonical bucket size:", len(canonical_bucket))

In [ ]:
list(redirect_bucket.items())[:20]

In [ ]:
list(canonical_bucket.items())[:10]

## 5. 遍历前几个 redirect pair

In [ ]:
pairs = []
with WikipediaRedirectIndex(INDEX_DIR) as index:
    for idx, pair in enumerate(index.iter_pairs()):
        pairs.append(pair)
        if idx >= 19:
            break

pairs

## 6. 写一个便于反复调用的小函数

In [ ]:
def inspect_title(title: str, synonym_limit: int = 20):
    normalized = normalize_wikipedia_title(title)
    bucket_key = get_bucket_key(normalized)
    with WikipediaRedirectIndex(INDEX_DIR) as index:
        canonical = index.resolve_redirect(title)
        synonyms = index.get_synonyms(title)
    return {
        "title": title,
        "normalized": normalized,
        "bucket_key": bucket_key,
        "canonical": canonical,
        "synonym_count": len(synonyms),
        "synonyms": synonyms[:synonym_limit],
    }

inspect_title("USA")